In [1]:
%reload_ext autoreload
%autoreload 2

In [ ]:
# --------------------- Import VQNiche ---------------------
from vqniche.utils.parse_test_configs import *
from vqniche.initializers.initialize import *
from vqniche import metrics
from vqniche.utils.type_conversions import *
from vqniche.utils.adjacency_reconstruction import reconstruct_adjacency_matrix as construct_binary_adjacency_matrix
from vqniche.plotting import *
from vqniche.utils.loss_utils import aggregate_1hop_neighbor_features

/software/cellgen/team361/am84/envs/vqniche-reproducibility/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/software/cellgen/team361/am84/envs/vqniche-reproducibility/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/software/cellgen/team361/am84/envs/vqniche-reproducibility/lib/python3.10/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain

In [ ]:
# --------------------- Import Libraries ---------------------
import os
import sys
import yaml
import pickle
from pathlib import Path

import scanpy as sc
import anndata as ad
import squidpy as sq

import numpy as np
import networkx as nx
import scipy.sparse as sp
from scipy.stats import pearsonr

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import pytorch_lightning as pl
import torch_geometric.transforms as T
from torch_geometric.data import Batch
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_adj

# --------------------- Display Settings ---------------------
# display setting all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Functions

### Initialize

In [5]:
def load_everything(
        config: Dict,
        model_ckpt_fname: Optional[str] = None,
    ):
    """
    Load all the necessary components for a model run.
    
    Parameters
    ----------
    config: Dict
        A dictionary containing the configuration parameters for the model run.
    
    Returns
    -------
    - None
    """
    # --------------------- Determinism Settings ---------------------
    pl.seed_everything(config['experiment']['seed'])

    # --------------------- Dataset ---------------------
    dataset_blob = initialize_dataset_blob(config)
    with open(Path(dataset_blob.processed_dir) / 'label_categories.pkl', 'rb') as f:
        label_categories = pickle.load(f)

    # --------------------- Databatch ---------------------
    data_batch = initialize_databatch(
                    config=config,
                    dataset_blob=dataset_blob,
                )
    
    # --------------------- Dataloader ---------------------
    datamodule_batch = initialize_datamodule(
                            config=config,
                            data=data_batch,
                        )

    # --------------------- Model ---------------------
    Model = set_model_class(config['model']['model_name'])
    if model_ckpt_fname is None:
        model_ckpt_fname = find_best_checkpoint(config['experiment']['wandb_run_dir'])
    model = Model.load_from_checkpoint(model_ckpt_fname)
    
    # --------------------- Trainer ---------------------
    strategy = "ddp_notebook"
    # strategy = "ddp"
    
    trainer = pl.Trainer(
                    accelerator="auto",
                    devices="auto",
                    deterministic=True,
                    logger=False,
                    callbacks=False,
                    strategy=strategy,
                    max_epochs=config['trainer']['max_epochs'],
                    enable_checkpointing=False,
                    num_sanity_val_steps=0,
                    enable_progress_bar=False,
                    enable_model_summary=True,
                )
    
    inference_data = model.collect_inference_data(
                    datamodule_batch.infer_dataloader()
                )
    inference_data['label_categories'] = label_categories
    
    return dataset_blob, \
        data_batch, \
        datamodule_batch, \
        model, \
        trainer, \
        inference_data

### Plotting

In [8]:
def plot_graph_spatial_layout(
        G: nx.Graph,
        G_hat: nx.Graph,
        label_list: list,
        label_key: str = 'cell_types',
        figsize: tuple = (12, 4),
        layout_algorithm: str = 'spring'
    ):
    """
    This function plots the layout of original and imputed graphs colored by the label.
    
    Parameters
    ----------
    G : nx.Graph
        Original networkx graph with node attributes.
    G_hat : nx.Graph
        Imputed networkx graph with node attributes.
    label_list : list
        List of labels corresponding to nodes (same order as G.nodes()).
    label_key : str
        Key in node attributes to use for coloring nodes.
    figsize : tuple
        Figure size (width, height).
    layout_algorithm : str
        Layout algorithm to use for positioning nodes.
        Options: 'spring', 'circular', 'random', 'shell', 'kamada_kawai'
    
    Returns
    -------
    None
    """
    # Add labels as node attributes to both graphs
    for i, node in enumerate(G.nodes()):
        if i < len(label_list):
            G.nodes[node][label_key] = label_list[i]
            G_hat.nodes[node][label_key] = label_list[i]
        else:
            # Handle case where there are more nodes than labels
            G.nodes[node][label_key] = 'Unknown'
            G_hat.nodes[node][label_key] = 'Unknown'
    
    # Create subplots
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # Get node labels for both graphs
    labels_G = nx.get_node_attributes(G, label_key)
    labels_G_hat = nx.get_node_attributes(G_hat, label_key)
    
    # Generate positions using layout algorithm
    if layout_algorithm == 'spring':
        pos_G = nx.spring_layout(G)
        pos_G_hat = nx.spring_layout(G_hat)
    elif layout_algorithm == 'circular':
        pos_G = nx.circular_layout(G)
        pos_G_hat = nx.circular_layout(G_hat)
    elif layout_algorithm == 'random':
        pos_G = nx.random_layout(G)
        pos_G_hat = nx.random_layout(G_hat)
    elif layout_algorithm == 'shell':
        pos_G = nx.shell_layout(G)
        pos_G_hat = nx.shell_layout(G_hat)
    elif layout_algorithm == 'kamada_kawai':
        pos_G = nx.kamada_kawai_layout(G)
        pos_G_hat = nx.kamada_kawai_layout(G_hat)
    else:
        pos_G = nx.spring_layout(G)
        pos_G_hat = nx.spring_layout(G_hat)
    
    # Get unique labels for consistent coloring
    all_labels = set(labels_G.values()) | set(labels_G_hat.values())
    unique_labels = sorted(list(all_labels))
    
    # Create color map
    colors = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))
    label_to_color = dict(zip(unique_labels, colors))
    
    # Plot original graph
    node_colors_G = [label_to_color.get(labels_G.get(node, 'Unknown'), 'gray') for node in G.nodes()]
    nx.draw(
        G, pos_G,
        node_color=node_colors_G,
        node_size=50,
        with_labels=False,
        ax=axes[0]
    )
    axes[0].set_title('Original Graph')
    
    # Plot imputed graph
    node_colors_G_hat = [label_to_color.get(labels_G_hat.get(node, 'Unknown'), 'gray') for node in G_hat.nodes()]
    nx.draw(
        G_hat, pos_G_hat,
        node_color=node_colors_G_hat,
        node_size=50,
        with_labels=False,
        ax=axes[1]
    )
    axes[1].set_title('Imputed Graph')
    
    # Create legend
    legend_elements = [plt.Line2D([0], [0], marker='o', color='w', 
                                  markerfacecolor=label_to_color[label], 
                                  markersize=10, label=label) 
                      for label in unique_labels]
    axes[1].legend(handles=legend_elements, title=label_key, loc='upper left', bbox_to_anchor=(1, 1))
    
    plt.tight_layout()
    plt.show()

# Analysis

## xhs1000-39b_1p (batch 11)

In [ ]:
# set dataset name and batch id
dataset_name = 'xhs1000-39b_1p'
batch_id = 11

### VQNiche

- Codebook Size: 5000
- Codebook Dim: 400
- Num GNN Layers: 1
- Encoder Conditions: RBF Distances
- Attribute Decoder Conditions: None
- Adjacency Decoder Conditions: None

In [ ]:
# set wandb run id
wandb_run_dir = Path('/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/logs/xhs1000-39b_1p/standalone/VQNiche/batch=[11]/spatial_n_neighs_8/wandb/run-20250724_103426-iah3xu4m/')

config = collect_test_configs(
    wandb_run_dir=wandb_run_dir,
)

pl.seed_everything(config['experiment']['seed'])

dataset_blob = initialize_dataset_blob(config)
with open(Path(dataset_blob.processed_dir) / 'label_categories.pkl', 'rb') as f:
    label_categories = pickle.load(f)

Model = set_model_class(config['model']['model_name'])
model = Model.load_from_checkpoint(
            config['model']['model_ckpt_fname'],
        )
model.eval()

results_dir = Path(config['experiment']['wandb_run_dir']) / 'results' / config['model']['model_ckpt_fname'].split('/')[-1].split('.')[0]
results_dir.mkdir(parents=True, exist_ok=True)

infer_dict_fname = results_dir / 'inference_data_dict.pkl'
with open(infer_dict_fname, 'rb') as f:
    inference_data = pickle.load(f)

adata = inference_data_dict_to_adata(
    inference_data=inference_data,
    label_categories_dict=label_categories,
)

Best checkpoint found: epoch=9-pearson_1hop_nbr=0.78.ckpt


In [ ]:
metrics_list = ['gcs', 'mlami', 'nasw']
metrics_values = metrics.compute_benchmarking_metrics(
                    adata=adata,
                    metrics=metrics_list,
                    cell_type_key='cell_type',
                    spatial_key='spatial',
                    latent_key='H_adj',
                    seed=42
                )
metrics_df = pd.DataFrame([
        {'metric': metric, 'score': score} 
        for metric, score in metrics_values.items()
    ])

display(metrics_df)

Using precomputed spatial nearest neighbor graph with 15 neighbors...
Using precomputed latent nearest neighbor graph with 15 neighbors...
Neighbor graphs computed. Elapsed time: 0 minutes 0 seconds.

Computing benchmarking metrics...
Computing MLAMI Metric...
Using precomputed spatial nearest neighbor graph...
Using precomputed latent nearest neighbor graph...
Computing spatial Leiden clusterings for entire dataset...
Computing latent Leiden clusterings...
Computing MLAMI for entire dataset...
MLAMI metric computed. Elapsed time: 0 minutes 26 seconds.

Computing GCS metric...
Using precomputed spatial nearest neighbor graph...
Using precomputed latent nearest neighbor graph...
Computing GCS for entire dataset...
GCS metric computed. Elapsed time: 0 minutes 26 seconds.

Computing NASW Metric...
Using precomputed latent nearest neighbor graph...
Computing latent Leiden clusterings...
Using precomputed latent Leiden clusters for resolution 0.1.
Using precomputed latent Leiden clusters fo

{'mlami': 0.19760299360996433,
 'gcs': 0.6758959494656902,
 'nasw': 0.4515385050326586}

In [ ]:
# compute UMAP embeddings for the original and imputed 1-hop neighbor attributes
embedding_keys = ['X', 'X_hat', 'X_nbr', 'X_hat_nbr']
adata.uns['X_nbr'] = aggregate_1hop_neighbor_features(
    X=adata.uns['X'],
    edge_index=adata.uns['edge_index'],
    return_mean=False,
)
adata.uns['X_hat_nbr'] = aggregate_1hop_neighbor_features(
    X=adata.uns['X_hat'],
    edge_index=adata.uns['edge_index'],
    return_mean=False,
)
adata = compute_umap(
        adata=adata,
        embedding_keys=embedding_keys,
    )

# plot UMAP embeddings for the original and imputed attributes colored by the cell types
plot_umap_attribute_imputation(
        adata=adata,
        embedding_keys=embedding_keys,
        label_key='cell_types',
    )

# plot UMAP embeddings for the original and imputed attributes colored by the niche types
plot_umap_attribute_imputation(
        adata=adata,
        embedding_keys=embedding_keys,
        label_key='niche_types',
    )

In [ ]:
# read the on_train_epoch_end_logs.csv file from the wandb run directory
df_fname = Path(config['experiment']['wandb_run_dir']) / 'files' / 'on_train_epoch_end_logs.csv'
df_loss, df_metrics = read_on_train_epoch_end_logs(
                            file_path=df_fname,
                        )

# plot the loss and metrics as a function of epoch
plot_logged_values_vs_epoch(
    df=df_loss,
    value_col="Value",
    name_col="Loss Term",
    mode_col="Mode",
    title="Losses vs Epoch",
)

# plot the metrics as a function of epoch
plot_logged_values_vs_epoch(
    df=df_metrics,
    value_col="Value",
    name_col="Metric",
    mode_col="Mode",
    title="Metrics vs Epoch",
)

In [ ]:
plot_code_assignment_on_xy_coordinates(
    adata=adata,
)